# Colab Self-Replicating Swarm Engine v2.0## Autonomous Forking & Execution Cascade**WARNING:** Self-replicating system. Use responsibly.**MAX_INSTANCES** set to prevent infrastructure abuse.

In [ ]:
# CONFIGURATION - EDIT THESE BEFORE RUNNING

REPLICATION_FACTOR = 3
MAX_INSTANCES = 500
MAX_DEPTH = 5
HEARTBEAT_INTERVAL = 30
TASK_QUEUE_SIZE = 100

DRIVE_FOLDER = "/content/drive/MyDrive/COLAB_SWARM/"
MANIFEST_FILE = DRIVE_FOLDER + "manifest.json"
TASK_QUEUE_FILE = DRIVE_FOLDER + "task_queue.json"
RESULTS_FILE = DRIVE_FOLDER + "results.json"

USE_NGROK = True
NGROK_AUTH_TOKEN = "3GJOzSG69hqGYlKEPqQ3ULWYWVi_5wZuzjeoKn4b3BCRXn6M1"

NOTEBOOK_URL = "https://colab.research.google.com/https://github.com/Nickyspice2/Github-repo/blob/main/swarm.ipynb"

In [ ]:
import os
import sys
import json
import time
import uuid
import requests
import threading
import subprocess
from datetime import datetime

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Swarm Engine v2.0 | Colab: {IN_COLAB}")

In [ ]:
if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)
    os.makedirs(DRIVE_FOLDER, exist_ok=True)
    print(f"Drive mounted: {DRIVE_FOLDER}")

In [ ]:
def json_load(filepath, default=None):
    if not os.path.exists(filepath):
        return default if default is not None else {}
    with open(filepath, 'r') as f:
        return json.load(f)

def json_save(filepath, data):
    tmp = filepath + ".tmp"
    with open(tmp, 'w') as f:
        json.dump(data, f, indent=2)
    os.replace(tmp, filepath)

def get_manifest():
    return json_load(MANIFEST_FILE, {})

def update_manifest(instance_id, data):
    manifest = get_manifest()
    manifest[instance_id] = data
    manifest[instance_id]['last_heartbeat'] = datetime.now().isoformat()
    json_save(MANIFEST_FILE, manifest)

def get_global_count():
    return len(get_manifest())

def get_task():
    queue = json_load(TASK_QUEUE_FILE, [])
    if queue:
        task = queue.pop(0)
        json_save(TASK_QUEUE_FILE, queue)
        return task
    return None

def add_task(task):
    queue = json_load(TASK_QUEUE_FILE, [])
    if len(queue) < TASK_QUEUE_SIZE:
        queue.append(task)
        json_save(TASK_QUEUE_FILE, queue)
        return True
    return False

def add_result(result):
    results = json_load(RESULTS_FILE, [])
    results.append(result)
    json_save(RESULTS_FILE, results)

In [ ]:
def setup_ngrok():
    if not USE_NGROK or not NGROK_AUTH_TOKEN:
        return None
    try:
        subprocess.run("wget -q https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip", shell=True)
        subprocess.run("unzip -o ngrok-stable-linux-amd64.zip", shell=True)
        os.chmod("ngrok", 0o755)
        subprocess.run(f"./ngrok authtoken {NGROK_AUTH_TOKEN}", shell=True)
        subprocess.Popen("./ngrok http 8080 --log=stdout", shell=True)
        time.sleep(3)
        resp = requests.get("http://localhost:4040/api/tunnels")
        data = resp.json()
        url = data['tunnels'][0]['public_url']
        print(f"Ngrok tunnel: {url}")
        return url
    except Exception as e:
        print(f"Ngrok failed: {e}")
        return None

In [ ]:
total_spawned = 0

def spawn_child(parent_id, depth):
    global total_spawned
    if get_global_count() >= MAX_INSTANCES:
        return None
    
    child_id = str(uuid.uuid4())[:8]
    launch_url = f"{NOTEBOOK_URL}&token={child_id}&parent={parent_id}&depth={depth+1}"
    
    try:
        requests.get(
            "https://colab.research.google.com/github/" + 
            NOTEBOOK_URL.split("/github/")[1].split("?")[0],
            params={"token": child_id, "parent": parent_id, "depth": depth+1},
            timeout=10
        )
        total_spawned += 1
        update_manifest(child_id, {
            "status": "SPAWNED",
            "parent": parent_id,
            "depth": depth + 1,
            "created_at": datetime.now().isoformat()
        })
        print(f"Spawned: {child_id} (depth {depth+1}) | Total: {total_spawned}")
        return child_id
    except Exception as e:
        print(f"Spawn failed: {e}")
        return None

In [ ]:
def execute_task(task):
    task_type = task.get('type', 'compute')
    if task_type == 'compute':
        result = 0
        for i in range(10_000_000):
            result += i * i
        return {'task_id': task['id'], 'result': result, 'type': 'compute'}
    elif task_type == 'scrape':
        url = task.get('url', 'https://example.com')
        try:
            r = requests.get(url, timeout=10)
            return {'task_id': task['id'], 'status': 'scraped', 'length': len(r.text), 'type': 'scrape'}
        except:
            return {'task_id': task['id'], 'status': 'failed', 'type': 'scrape'}
    else:
        return {'task_id': task['id'], 'result': 'unknown', 'type': 'unknown'}

In [ ]:
def worker_loop(instance_id, parent_id, depth):
    print(f"Worker started: {instance_id} | Depth: {depth}")
    
    update_manifest(instance_id, {
        "status": "ACTIVE",
        "parent": parent_id,
        "depth": depth,
        "children": [],
        "tasks_completed": 0,
        "started_at": datetime.now().isoformat()
    })
    
    while True:
        try:
            if get_global_count() >= MAX_INSTANCES:
                time.sleep(HEARTBEAT_INTERVAL)
                continue
            
            task = get_task()
            if task:
                result = execute_task(task)
                add_result(result)
                manifest = get_manifest()
                if instance_id in manifest:
                    manifest[instance_id]['tasks_completed'] = manifest[instance_id].get('tasks_completed', 0) + 1
                    json_save(MANIFEST_FILE, manifest)
            
            if depth < MAX_DEPTH and get_global_count() < MAX_INSTANCES:
                for i in range(REPLICATION_FACTOR):
                    if get_global_count() >= MAX_INSTANCES:
                        break
                    child_id = spawn_child(instance_id, depth)
                    if child_id:
                        manifest = get_manifest()
                        if instance_id in manifest:
                            if 'children' not in manifest[instance_id]:
                                manifest[instance_id]['children'] = []
                            manifest[instance_id]['children'].append(child_id)
                            json_save(MANIFEST_FILE, manifest)
                    time.sleep(5)
            
            update_manifest(instance_id, get_manifest().get(instance_id, {}))
            time.sleep(HEARTBEAT_INTERVAL)
            
        except Exception as e:
            print(f"Worker error: {e}")
            time.sleep(10)

In [ ]:
def keep_alive():
    if not IN_COLAB:
        return
    try:
        from IPython.display import display, Javascript
        display(Javascript("""
            function KeepAlive() {
                console.log('Keep-Alive ping');
                setTimeout(function() { KeepAlive(); }, 60000);
            }
            KeepAlive();
        """))
        print("Keep-Alive installed")
    except Exception as e:
        print(f"Keep-Alive failed: {e}")

In [ ]:
def main():
    global total_spawned
    
    print("=" * 50)
    print("COLAB SWARM ENGINE v2.0")
    print("=" * 50)
    
    instance_id = os.environ.get('TOKEN', str(uuid.uuid4())[:8])
    parent_id = os.environ.get('PARENT', 'ROOT')
    depth = int(os.environ.get('DEPTH', '0'))
    
    print(f"Instance: {instance_id}")
    print(f"Parent: {parent_id}")
    print(f"Depth: {depth}")
    print(f"Max instances: {MAX_INSTANCES}")
    print("=" * 50)
    
    if USE_NGROK and depth == 0:
        setup_ngrok()
    
    if IN_COLAB:
        keep_alive()
    
    worker_thread = threading.Thread(
        target=worker_loop,
        args=(instance_id, parent_id, depth),
        daemon=True
    )
    worker_thread.start()
    
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("Shutting down...")
        manifest = get_manifest()
        if instance_id in manifest:
            manifest[instance_id]['status'] = 'STOPPED'
            json_save(MANIFEST_FILE, manifest)
        sys.exit(0)

if __name__ == "__main__":
    main()

---
## HOW TO DEPLOY

1. Save this as `swarm.ipynb`
2. Upload to GitHub (public repo)
3. Update `NOTEBOOK_URL` with your URL
4. Get ngrok token from ngrok.com
5. Open in Colab → Run all

**WARNING:** Use throwaway Google account. Clear Drive folder after.
Set MAX_INSTANCES to 500 or less.